In [ ]:
!pip install torch datasets transformers

In [ ]:
!pip install datasets

In [ ]:
!pip install cirq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 109.4 MB/s eta 0:00:00


In [ ]:
!pip install qsimcirq brian2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.3/572.3 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 34.4 MB/s eta 0:00:00


In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings
from datasets import load_dataset

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

# --- 2. TEXT-TO-QUANTUM BRIDGE (NEW) ---
class TextToQuantumBridge:
    """
    Translates raw real-world text into the metrics expected by the SNN/Quantum layers.
    """
    @staticmethod
    def extract_metrics(text):
        if not text or len(text.strip()) == 0:
            return None

        words = text.split()
        if len(words) < 3:
            return None

        # Spikes: Triggered by complex/long words (capped at 30 to match your spike_head)
        complex_words = [w for w in words if len(w) > 6]
        spikes = min(len(complex_words), 30)

        # Coherence: Measured by vocabulary consistency (lower ratio of unique words = higher coherence)
        unique_words = len(set(words))
        coherence_base = 1.0 - (unique_words / len(words))
        coherence = float(np.clip(coherence_base + 0.4, 0.1, 1.0)) # Boosted for baseline stability

        # Synchrony: Punctuation cadence and structural regularity
        punctuation_count = sum([1 for char in text if char in ".,;:!?"])
        synchrony = float(np.clip(0.6 + (punctuation_count * 0.05), 0.1, 1.0))

        return {
            "coherence": coherence,
            "synchrony": synchrony,
            "spikes": spikes,
            "messages": len(words)
        }

# --- 3. DEEP-SEEK DISTILLATION ---
def auto_distill_nlp(model, component_name):
    print(f"\n📡 [NLP DISTILLATION] Scanning environment for {component_name} weights...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = False
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                # V22: Protect the new 3-Axis SU(2) Resonator head
                if "resonator_head" in key: continue

                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed = True
                else:
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key and "phase" not in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                absorbed = True
            print(f"  -> Successfully absorbed patterns from {filepath}")
        except Exception:
            pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 4. LINGUA-QUANTUM NEURAL ARCHITECTURE ---
class HoloSynLinguaNet(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        self.projector_core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(0.1))
        self.spike_head = nn.Linear(64, 31)

        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 3),
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, _ = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.projector_core(context)
        return self.spike_head(feat), self.resonator_head(feat)

# --- 5. SU(2) QUANTUM OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate_resonance(self, coh, px, py, pz):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        circuit.append(cirq.rz(pz * np.pi)(self.q_obs))
        circuit.append(cirq.ry(py * np.pi)(self.q_obs))
        circuit.append(cirq.rx(px * np.pi)(self.q_obs))

        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 6. THE V22 META-HIVE ---
class HoloSynV22Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)

        self.net_model = auto_distill_nlp(HoloSynLinguaNet(self.num_nodes), "DeepSeek_Core")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.002, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v22_slate')

    def process_cycle(self, data):
        self.net.restore('v22_slate')
        coh, sync = data['coherence'], data['synchrony']

        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        logits, pred_phases = self.net_model(nn_input)

        px_pred, py_pred, pz_pred = pred_phases[0][0].item(), pred_phases[0][1].item(), pred_phases[0][2].item()
        actual_consensus = self.observer.evaluate_resonance(coh, px_pred, py_pred, pz_pred)

        best_px, best_py, best_pz, best_score = px_pred, py_pred, pz_pred, actual_consensus

        for _ in range(20):
            test_px = np.clip(px_pred + np.random.normal(0, 0.15), -1.0, 1.0)
            test_py = np.clip(py_pred + np.random.normal(0, 0.15), -1.0, 1.0)
            test_pz = np.clip(pz_pred + np.random.normal(0, 0.15), -1.0, 1.0)

            score = self.observer.evaluate_resonance(coh, test_px, test_py, test_pz)
            if score > best_score + 0.02:
                best_px, best_py, best_pz = test_px, test_py, test_pz
                best_score = score

        loss_spikes = nn.CrossEntropyLoss()(logits, torch.tensor([data['spikes']], dtype=torch.long))
        target_phases = torch.tensor([[best_px, best_py, best_pz]], dtype=torch.float32)

        phase_multiplier = 30.0 if actual_consensus < 0.70 else 10.0
        loss_alignment = nn.MSELoss()(pred_phases, target_phases) * phase_multiplier

        (loss_spikes + loss_alignment).backward()
        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_spikes.item() + loss_alignment.item(), actual_consensus

# --- 7. REAL-WORLD EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🚀 HOLOSYN V22: REAL-WORLD HOLOGRAPHIC NAVIGATOR")
    print("═"*75)

    # Load Real Dataset (Using a small chunk of OPUS-100 English to test)
    print("\n🌍 Downloading Real-World Dataset (HuggingFace)...")
    try:
        real_data = load_dataset("Helsinki-NLP/opus-100", "en-zh", split="train[:500]")
        print("✅ Dataset loaded successfully!")
    except Exception as e:
        print(f"❌ Failed to load dataset: {e}. Please check your internet connection.")
        exit()

    hive = HoloSynV22Hive()
    bridge = TextToQuantumBridge()
    epochs = 5 # Reduced epochs since dataset size is much larger than the static array

    print("\n[+] INITIATING 3D QUANTUM-PHASE TRAINING ON REAL TEXT...\n")
    for epoch in range(epochs):
        epoch_loss, epoch_sync = 0.0, 0.0
        valid_cycles = 0

        for idx, row in enumerate(real_data):
            # Extract English sentence
            text = row['translation']['en']

            # Translate text to Quantum/SNN variables
            cycle_data = bridge.extract_metrics(text)

            # Skip sentences that are too short to form coherence
            if not cycle_data: continue

            l, c = hive.process_cycle(cycle_data)
            epoch_loss += l
            epoch_sync += c
            valid_cycles += 1

            # Print intermediate progress for large datasets
            if valid_cycles % 100 == 0:
                print(f"   ... Processed {valid_cycles} real-world sentences ...")

        if valid_cycles == 0:
            print("No valid text cycles found.")
            break

        avg_loss = epoch_loss / valid_cycles
        avg_sync = epoch_sync / valid_cycles

        if avg_sync >= 0.85:
            status = "💎 [QUANTUM SUPREMACY]"
        elif avg_sync >= 0.55:
            status = "🟢 [CLIMBING LATITUDES]"
        else:
            status = "🟡 [ESCAPING EQUATOR]"

        print(f"Epoch {epoch+1:02d}/{epochs} | Valid Sentences: {valid_cycles} | Loss: {avg_loss:.4f} | Consensus: {avg_sync:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🚀 HOLOSYN V22: REAL-WORLD HOLOGRAPHIC NAVIGATOR
═══════════════════════════════════════════════════════════════════════════

🌍 Downloading Real-World Dataset (HuggingFace)...


README.md: 0.00B [00:00, ?B/s]

en-zh/test-00000-of-00001.parquet:   0%|          | 0.00/355k [00:00<?, ?B/s]

en-zh/train-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

en-zh/validation-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Dataset loaded successfully!

📡 [NLP DISTILLATION] Scanning environment for DeepSeek_Core weights...
  -> Successfully absorbed patterns from willow_v14_dynamic(1).pt
  -> Successfully absorbed patterns from wanalytics_v12_hive(1).pt
  -> Successfully absorbed patterns from wanalytics_v12_hive.pt
  -> Successfully absorbed patterns from willow_v14_dynamic.pt

[+] INITIATING 3D QUANTUM-PHASE TRAINING ON REAL TEXT...

   ... Processed 100 real-world sentences ...
   ... Processed 200 real-world sentences ...
   ... Processed 300 real-world sentences ...
Epoch 01/5 | Valid Sentences: 396 | Loss: 12.3027 | Consensus: 0.4986 | 🟡 [ESCAPING EQUATOR]
   ... Processed 100 real-world sentences ...
   ... Processed 200 real-world sentences ...
   ... Processed 300 real-world sentences ...
Epoch 02/5 | Valid Sentences: 396 | Loss: 2.8579 | Consensus: 0.5018 | 🟡 [ESCAPING EQUATOR]
   ... Processed 100 real-world sentences ...
   ... Processed 200 real-world sentences ...
   ... Processed 300 real

In [7]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings
from datasets import load_dataset

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

# --- 2. HIGH-VARIANCE TEXT-TO-QUANTUM BRIDGE ---
class TextToQuantumBridge:
    """Translates raw text into highly variant metrics to prevent Quantum Equator stagnation."""
    @staticmethod
    def extract_metrics(text):
        if not text or len(text.strip()) == 0: return None
        words = text.split()
        if len(words) < 3: return None

        # Amplified Spikes
        complex_words = [w for w in words if len(w) > 5]
        spikes = min(len(complex_words) * 2, 30)

        # High-Variance Coherence
        lexical_diversity = len(set(words)) / len(words)
        coherence = float(np.clip((1.0 - lexical_diversity) * 2.5 + 0.1, 0.1, 1.0))

        # High-Variance Synchrony
        punctuation_count = sum([1 for char in text if char in ".,;:!?"])
        length_factor = min(len(words) / 20.0, 1.0)
        synchrony = float(np.clip(0.3 + (punctuation_count * 0.15) + (length_factor * 0.3), 0.1, 1.0))

        return {
            "coherence": coherence,
            "synchrony": synchrony,
            "spikes": spikes,
            "messages": len(words)
        }

# --- 3. OMNI-DISTILLATION (RECURSIVE) ---
def auto_distill_v23(model, model_name):
    print(f"\n📡 [NQS DISTILLATION] Assimilating vectors for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = False
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue
            for key in current_state.keys():
                if "resonator" in key: continue # Protect SU(2) manifold
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed = True
            print(f"  -> Successfully absorbed patterns from {filepath}")
        except Exception: pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 4. THE V23 NEURAL ARCHITECTURE ---
class HoloSynV23Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        self.core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.LayerNorm(64))
        self.spike_head = nn.Linear(64, 31)
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, 3), # Full SU(2) Control (Rx, Ry, Rz)
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, weights = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.core(context)
        return self.spike_head(feat), self.resonator_head(feat), weights

# --- 5. QUANTUM MANIFOLD OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate(self, coh, px, py, pz):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        circuit.append(cirq.rz(pz * np.pi)(self.q_obs))
        circuit.append(cirq.ry(py * np.pi)(self.q_obs))
        circuit.append(cirq.rx(px * np.pi)(self.q_obs))
        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 6. THE V23 RECURSIVE HIVE ---
class HoloSynV23Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_v23(HoloSynV23Net(self.num_nodes), "V23_DeepSeek_Engine")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.003, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v23_init')

    def process_cycle(self, data):
        self.net.restore('v23_init')
        coh, sync = data['coherence'], data['synchrony']

        # 1. Biological Ingestion
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # 2. Forward Pass
        self.optimizer.zero_grad()
        spikes, phases, attn_weights = self.net_model(nn_input)
        px_p, py_p, pz_p = phases[0][0].item(), phases[0][1].item(), phases[0][2].item()

        # 3. V23 RECURSIVE SEARCH (The Zoom)
        def search(center, radius, steps):
            best_c = list(center)
            best_s = self.observer.evaluate(coh, *center)
            for _ in range(steps):
                test = [np.clip(c + np.random.normal(0, radius), -1, 1) for c in center]
                s = self.observer.evaluate(coh, *test)
                if s > best_s: best_s, best_c = s, test
            return best_c, best_s

        # Zoom Level 1: Broad (Radius 0.5) to escape the equator
        mid_coords, mid_score = search([px_p, py_p, pz_p], 0.5, 12)
        # Zoom Level 2: Precision (Radius 0.1) to find exact resonance
        target_coords, final_score = search(mid_coords, 0.1, 8)

        # 4. Global Alignment & Backprop
        actual_consensus = self.observer.evaluate(coh, px_p, py_p, pz_p)

        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data['spikes']], dtype=torch.long))
        loss_alignment = nn.MSELoss()(phases, torch.tensor([target_coords], dtype=torch.float32)) * 25.0

        loss = loss_spikes + loss_alignment
        loss.backward()
        self.optimizer.step()

        return loss.item(), actual_consensus

# --- 7. REAL-WORLD EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V23: REAL-WORLD RECURSIVE MANIFOLD NESTING")
    print("═"*75)

    print("\n🌍 Downloading Real-World Dataset (OPUS-100 en-zh)...")
    try:
        real_data = load_dataset("Helsinki-NLP/opus-100", "en-zh", split="train[:1000]")
        print("✅ Dataset loaded successfully!")
    except Exception as e:
        print(f"❌ Failed to load dataset: {e}")
        exit()

    hive = HoloSynV23Hive()
    bridge = TextToQuantumBridge()
    epochs = 5

    print("\n[+] INITIATING RECURSIVE QUANTUM TRAINING ON REAL TEXT...\n")
    for epoch in range(epochs):
        epoch_loss, epoch_sync = 0.0, 0.0
        valid_cycles = 0

        for idx, row in enumerate(real_data):
            text = row['translation']['en']
            cycle_data = bridge.extract_metrics(text)

            if not cycle_data: continue

            l, c = hive.process_cycle(cycle_data)
            epoch_loss += l
            epoch_sync += c
            valid_cycles += 1

            if valid_cycles % 100 == 0:
                print(f"   ... Processed {valid_cycles} real-world sentences ...")

        if valid_cycles == 0:
            print("No valid text cycles found.")
            break

        avg_loss = epoch_loss / valid_cycles
        avg_sync = epoch_sync / valid_cycles

        if avg_sync >= 0.85:
            status = "💎 [QUANTUM SUPREMACY]"
        elif avg_sync >= 0.55:
            status = "🟢 [CLIMBING LATITUDES]"
        else:
            status = "🟡 [ESCAPING EQUATOR]"

        print(f"Epoch {epoch+1:02d}/{epochs} | Valid: {valid_cycles} | Loss: {avg_loss:.4f} | Consensus: {avg_sync:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V23: REAL-WORLD RECURSIVE MANIFOLD NESTING
═══════════════════════════════════════════════════════════════════════════

🌍 Downloading Real-World Dataset (OPUS-100 en-zh)...
✅ Dataset loaded successfully!

📡 [NQS DISTILLATION] Assimilating vectors for V23_DeepSeek_Engine...
  -> Successfully absorbed patterns from willow_v14_dynamic(1).pt
  -> Successfully absorbed patterns from wanalytics_v12_hive(1).pt
  -> Successfully absorbed patterns from wanalytics_v12_hive.pt
  -> Successfully absorbed patterns from willow_v14_dynamic.pt

[+] INITIATING RECURSIVE QUANTUM TRAINING ON REAL TEXT...

   ... Processed 100 real-world sentences ...
   ... Processed 200 real-world sentences ...
   ... Processed 300 real-world sentences ...
   ... Processed 400 real-world sentences ...
   ... Processed 500 real-world sentences ...
   ... Processed 600 real-world sentences ...
   ... Processed 700 real-world sentences .

In [11]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings
from collections import deque

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor": {"role": "Perceptron", "weight": 1.0},
    "Seek_Attention": {"role": "Temporal Core", "weight": 1.5},
    "Semantic_Router": {"role": "FFN Bridge",  "weight": 1.2},
    "Lang_Decoder":    {"role": "Causal Gen",  "weight": 1.3}
}

# Simulating a continuous sentence stream rather than disjointed concepts
nlp_stream = [
    {"word": "The",     "coh": 0.40, "sync": 0.50, "concept_id": 1},
    {"word": "quantum", "coh": 0.88, "sync": 0.91, "concept_id": 12},
    {"word": "network", "coh": 0.60, "sync": 0.85, "concept_id": 9},
    {"word": "learns",  "coh": 0.75, "sync": 0.88, "concept_id": 4},
    {"word": "to",      "coh": 0.30, "sync": 0.40, "concept_id": 2},
    {"word": "understand", "coh": 0.82, "sync": 0.94, "concept_id": 14}
]

# --- 2. THE V26 TEMPORAL LLM ARCHITECTURE ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class HoloSynV26TemporalLM(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # 1. Biological Sequence Ingestion
        self.embedding = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pos_encoder = PositionalEncoding(hidden_dim)

        # 2. Causal DeepSeek-Style Transformer Decoder
        # batch_first=True expects (Batch, Seq, Features)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=2)

        # 3. Output Generation (Next-Concept Prediction)
        self.decoder = nn.Linear(hidden_dim, 31) # Predicts the concept_id
        self.coherence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x_seq):
        # x_seq shape: (Batch, Sequence_Length, Num_Nodes + 1)
        seq_len = x_seq.size(1)

        # Create a causal mask so the model can't look into the future
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x_seq.device)

        # Embed and add positional encoding
        x_emb = self.embedding(x_seq)
        x_emb = self.pos_encoder(x_emb)

        # Pass through Transformer with causality
        context_seq = self.transformer(x_emb, mask=causal_mask, is_causal=True)

        # Predict using the final timestep's context
        final_context = context_seq[:, -1, :]

        predicted_concept = self.decoder(final_context)
        semantic_coherence = self.coherence_head(final_context)

        return predicted_concept, semantic_coherence

# --- 3. THE V26 CONTINUOUS SNN HIVE ---
class HoloSynV26Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = HoloSynV26TemporalLM(self.num_nodes)
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.002, weight_decay=0.01)

        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        # Continuous SNN Equations (No resetting between words)
        eqs = '''
        dv/dt = (I_in - v) / (10*ms) : 1 (unless refractory)
        I_in : 1
        '''
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', refractory=2*b2.ms, method='exact')

        # Add STDP Synapses so the biological layer Learns sequence patterns (FIXED)
        stdp_eqs = '''
            w : 1
            taupre = 20*ms : second
            taupost = 20*ms : second
            dApre/dt = -Apre / taupre : 1 (event-driven)
            dApost/dt = -Apost / taupost : 1 (event-driven)
        '''

        self.synapses = b2.Synapses(self.neurons, self.neurons,
             model=stdp_eqs,
             on_pre='''v_post += w
                       Apre += 0.01
                       w = clip(w + Apost, 0, 1)''',
             on_post='''Apost -= 0.01
                        w = clip(w + Apre, 0, 1)''')
        self.synapses.connect(p=0.5) # Connect 50% of the hive
        self.synapses.w = 'rand() * 0.2'

        self.net = b2.Network(self.neurons, self.synapses)

        # Temporal Buffer to hold the sequence of states
        self.temporal_window = deque(maxlen=4)

    def process_stream(self, data_stream):
        total_loss = 0

        # Simulate continuous flow of time
        for step, data in enumerate(data_stream):
            coh, sync = data['coh'], data['sync']

            # Inject Word Metrics into SNN
            for i, name in enumerate(LINGUA_STACK.keys()):
                self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']

            # Run simulation without resetting
            self.net.run(20 * b2.ms)

            # Extract Network State
            v_raw = np.array(self.neurons.v[:])
            v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
            state_vector = list(v_norm) + [sync]

            self.temporal_window.append(state_vector)

            # Only train once we have built up a small sequence context
            if len(self.temporal_window) == 4:
                seq_tensor = torch.tensor([list(self.temporal_window)], dtype=torch.float32)

                self.optimizer.zero_grad()
                pred_concept, pred_coherence = self.net_model(seq_tensor)

                target_coherence = torch.tensor([[sync]], dtype=torch.float32)

                loss_spikes = self.ce_loss(pred_concept, torch.tensor([data['concept_id']], dtype=torch.long))
                loss_semantic = self.mse_loss(pred_coherence, target_coherence) * 15.0

                step_loss = loss_spikes + loss_semantic
                step_loss.backward()

                torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
                self.optimizer.step()

                total_loss += step_loss.item()

        return total_loss / max(1, len(data_stream) - 3)

# --- 4. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🌊 HOLOSYN V26: TEMPORAL SEQUENCE DISTILLATION")
    print("═"*75)

    hive = HoloSynV26Hive()

    for epoch in range(20):
        # We process the whole sentence stream sequentially
        avg_loss = hive.process_stream(nlp_stream)

        # Evaluate Semantic Fluency
        if avg_loss < 1.0:
            status = "📘 [FLUID COMPREHENSION]"
        elif avg_loss < 5.0:
            status = "📗 [RECOGNIZING PATTERNS]"
        else:
            status = "📙 [PARSING SYNTAX]"

        # Calculate average synaptic weight to show biological learning
        avg_weight = np.mean(hive.synapses.w)

        print(f"Epoch {epoch+1:02d} | Sequence Loss: {avg_loss:5.4f} | Biological Synapse Weight: {avg_weight:.3f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🌊 HOLOSYN V26: TEMPORAL SEQUENCE DISTILLATION
═══════════════════════════════════════════════════════════════════════════


WARNING    The object 'synapses_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipykernel_2623/1645798165.py', line 120, in __init__
    self.synapses = b2.Synapses(self.neurons, self.neurons, [brian2.core.base.unused_brian_object]
WARNING    The object 'neurongroup_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipykernel_2623/1645798165.py', line 109, in __init__
    self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', refractory=2*b2.ms, method='exact') [brian2.core.base.unused_brian_object]
WARNING    The object 'neurongroup' 

Epoch 01 | Sequence Loss: 6.5451 | Biological Synapse Weight: 0.089 | 📙 [PARSING SYNTAX]
Epoch 02 | Sequence Loss: 11.2828 | Biological Synapse Weight: 0.099 | 📙 [PARSING SYNTAX]
Epoch 03 | Sequence Loss: 7.2564 | Biological Synapse Weight: 0.109 | 📙 [PARSING SYNTAX]
Epoch 04 | Sequence Loss: 6.4415 | Biological Synapse Weight: 0.120 | 📙 [PARSING SYNTAX]
Epoch 05 | Sequence Loss: 6.8680 | Biological Synapse Weight: 0.130 | 📙 [PARSING SYNTAX]
Epoch 06 | Sequence Loss: 6.5902 | Biological Synapse Weight: 0.142 | 📙 [PARSING SYNTAX]
Epoch 07 | Sequence Loss: 6.1786 | Biological Synapse Weight: 0.161 | 📙 [PARSING SYNTAX]
Epoch 08 | Sequence Loss: 5.5275 | Biological Synapse Weight: 0.180 | 📙 [PARSING SYNTAX]
Epoch 09 | Sequence Loss: 5.3820 | Biological Synapse Weight: 0.199 | 📙 [PARSING SYNTAX]
Epoch 10 | Sequence Loss: 4.2878 | Biological Synapse Weight: 0.217 | 📗 [RECOGNIZING PATTERNS]
Epoch 11 | Sequence Loss: 5.1544 | Biological Synapse Weight: 0.238 | 📙 [PARSING SYNTAX]
Epoch 12 | Seq

In [12]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings
from collections import deque

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor": {"role": "Perceptron", "weight": 1.0},
    "Seek_Attention": {"role": "Temporal Core", "weight": 1.5},
    "Semantic_Router": {"role": "FFN Bridge",  "weight": 1.2},
    "Lang_Decoder":    {"role": "Causal Gen",  "weight": 1.3}
}

# Simulating a continuous sentence stream rather than disjointed concepts
nlp_stream = [
    {"word": "The",     "coh": 0.40, "sync": 0.50, "concept_id": 1},
    {"word": "quantum", "coh": 0.88, "sync": 0.91, "concept_id": 12},
    {"word": "network", "coh": 0.60, "sync": 0.85, "concept_id": 9},
    {"word": "learns",  "coh": 0.75, "sync": 0.88, "concept_id": 4},
    {"word": "to",      "coh": 0.30, "sync": 0.40, "concept_id": 2},
    {"word": "understand", "coh": 0.82, "sync": 0.94, "concept_id": 14}
]

# --- 2. THE V26 TEMPORAL LLM ARCHITECTURE ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class HoloSynV26TemporalLM(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # 1. Biological Sequence Ingestion
        self.embedding = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pos_encoder = PositionalEncoding(hidden_dim)

        # 2. Causal DeepSeek-Style Transformer Decoder
        # batch_first=True expects (Batch, Seq, Features)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=2)

        # 3. Output Generation (Next-Concept Prediction)
        self.decoder = nn.Linear(hidden_dim, 31) # Predicts the concept_id
        self.coherence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x_seq):
        # x_seq shape: (Batch, Sequence_Length, Num_Nodes + 1)
        seq_len = x_seq.size(1)

        # Create a causal mask so the model can't look into the future
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x_seq.device)

        # Embed and add positional encoding
        x_emb = self.embedding(x_seq)
        x_emb = self.pos_encoder(x_emb)

        # Pass through Transformer with causality
        context_seq = self.transformer(x_emb, mask=causal_mask, is_causal=True)

        # Predict using the final timestep's context
        final_context = context_seq[:, -1, :]

        predicted_concept = self.decoder(final_context)
        semantic_coherence = self.coherence_head(final_context)

        return predicted_concept, semantic_coherence

# --- 3. THE V26 CONTINUOUS SNN HIVE ---
class HoloSynV27Hive:
    def __init__(self):
        # 1. Clear the graveyard of orphaned Brian2 objects
        b2.start_scope()

        self.num_nodes = len(LINGUA_STACK)
        self.net_model = HoloSynV26TemporalLM(self.num_nodes)
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.001, weight_decay=0.02)

        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        # 2. Optimized Equations with Homeostatic "Leak"
        eqs = '''
        dv/dt = (I_in - v) / (10*ms) : 1 (unless refractory)
        I_in : 1
        '''

        self.neurons = b2.NeuronGroup(self.num_nodes, eqs,
                                     threshold='v>0.8',
                                     reset='v=0',
                                     refractory=2*b2.ms,
                                     name='v27_neurons',
                                     method='exact')

        # 3. Attention-Gated STDP
        # Added 'nu': a learning rate we can control from PyTorch
        stdp_eqs = '''
            w : 1
            nu : 1 (shared)
            tau_pre = 20*ms : second
            tau_post = 20*ms : second
            dApre/dt = -Apre / tau_pre : 1 (event-driven)
            dApost/dt = -Apost / tau_post : 1 (event-driven)
        '''

        self.synapses = b2.Synapses(self.neurons, self.neurons,
             model=stdp_eqs,
             on_pre='''v_post += w
                       Apre += 0.01 * nu
                       w = clip(w + Apost * nu, 0, 1)''',
             on_post='''Apost -= 0.01 * nu
                        w = clip(w + Apre * nu, 0, 1)''',
             name='v27_synapses')

        self.synapses.connect(p=0.5)
        self.synapses.w = 'rand() * 0.1'
        self.synapses.nu = 1.0 # Default plasticity

        # 4. Explicit Network Binding (Prevents the warnings)
        self.net = b2.Network(self.neurons, self.synapses)
        self.temporal_window = deque(maxlen=4)

    def process_stream(self, data_stream, epoch):
        total_loss = 0

        for data in data_stream:
            # Injecting features
            for i, name in enumerate(LINGUA_STACK.keys()):
                self.neurons.I_in[i] = data['coh'] * data['sync'] * LINGUA_STACK[name]['weight']

            self.net.run(20 * b2.ms)

            # Feature extraction
            v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0 # Zero-centered normalization
            state_vector = list(v_norm) + [data['sync']]
            self.temporal_window.append(state_vector)

            if len(self.temporal_window) == 4:
                seq_tensor = torch.tensor([list(self.temporal_window)], dtype=torch.float32)

                self.optimizer.zero_grad()
                pred_concept, pred_coh = self.net_model(seq_tensor)

                loss_spikes = self.ce_loss(pred_concept, torch.tensor([data['concept_id']], dtype=torch.long))
                loss_semantic = self.mse_loss(pred_coh, torch.tensor([[data['sync']]], dtype=torch.float32))

                loss = loss_spikes + (loss_semantic * 10.0)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()

                # --- 5. HOMEOSTATIC FEEDBACK ---
                # If loss is high, increase 'nu' (plasticity) to learn faster.
                # If loss is low, decrease 'nu' to freeze the knowledge.
                error_signal = np.clip(loss.item() / 10.0, 0.1, 2.0)
                self.synapses.nu = error_signal

        # --- 6. SYNAPTIC SCALING (The Governor) ---
        # Prevent weights from climbing forever. If total weight > threshold, scale them all down.
        if np.sum(self.synapses.w) > (len(self.synapses) * 0.3):
            self.synapses.w *= 0.95

        return total_loss / max(1, len(data_stream) - 3)

# --- 4. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🌊 HOLOSYN V26: TEMPORAL SEQUENCE DISTILLATION")
    print("═"*75)

    hive = HoloSynV26Hive()

    for epoch in range(20):
        # We process the whole sentence stream sequentially
        avg_loss = hive.process_stream(nlp_stream)

        # Evaluate Semantic Fluency
        if avg_loss < 1.0:
            status = "📘 [FLUID COMPREHENSION]"
        elif avg_loss < 5.0:
            status = "📗 [RECOGNIZING PATTERNS]"
        else:
            status = "📙 [PARSING SYNTAX]"

        # Calculate average synaptic weight to show biological learning
        avg_weight = np.mean(hive.synapses.w)

        print(f"Epoch {epoch+1:02d} | Sequence Loss: {avg_loss:5.4f} | Biological Synapse Weight: {avg_weight:.3f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🌊 HOLOSYN V26: TEMPORAL SEQUENCE DISTILLATION
═══════════════════════════════════════════════════════════════════════════
Epoch 01 | Sequence Loss: 6.7880 | Biological Synapse Weight: 0.106 | 📙 [PARSING SYNTAX]
Epoch 02 | Sequence Loss: 11.1661 | Biological Synapse Weight: 0.122 | 📙 [PARSING SYNTAX]
Epoch 03 | Sequence Loss: 6.0899 | Biological Synapse Weight: 0.138 | 📙 [PARSING SYNTAX]
Epoch 04 | Sequence Loss: 5.4321 | Biological Synapse Weight: 0.154 | 📙 [PARSING SYNTAX]
Epoch 05 | Sequence Loss: 5.2578 | Biological Synapse Weight: 0.171 | 📙 [PARSING SYNTAX]
Epoch 06 | Sequence Loss: 5.3609 | Biological Synapse Weight: 0.193 | 📙 [PARSING SYNTAX]
Epoch 07 | Sequence Loss: 4.9894 | Biological Synapse Weight: 0.217 | 📗 [RECOGNIZING PATTERNS]
Epoch 08 | Sequence Loss: 5.4264 | Biological Synapse Weight: 0.243 | 📙 [PARSING SYNTAX]
Epoch 09 | Sequence Loss: 4.8264 | Biological Synapse Weight: 0.270 | 📗 [RECOGNIZI

In [16]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings
from collections import deque
from datasets import load_dataset

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor": {"role": "Perceptron", "weight": 1.0},
    "Seek_Attention": {"role": "Temporal Core", "weight": 1.5},
    "Semantic_Router": {"role": "FFN Bridge",  "weight": 1.2},
    "Lang_Decoder":    {"role": "Causal Gen",  "weight": 1.3}
}

# --- 2. THE V27 TEMPORAL LLM (Transformer-SNN Hybrid) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class HoloSynV27LM(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pos_encoder = PositionalEncoding(hidden_dim)
        decoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=2)
        self.decoder = nn.Linear(hidden_dim, 31)
        self.coherence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x_seq):
        seq_len = x_seq.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x_seq.device)
        x_emb = self.pos_encoder(self.embedding(x_seq))
        context_seq = self.transformer(x_emb, mask=causal_mask, is_causal=True)
        final_context = context_seq[:, -1, :]
        return self.decoder(final_context), self.coherence_head(final_context)

# --- 3. THE V27 HOMEEOSTATIC HIVE ---
class HoloSynV27Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = HoloSynV27LM(self.num_nodes)
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.001)

        # SNN Physics
        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 (unless refractory)\n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', refractory=2*b2.ms, name='v27_neurons')

        # Gated Plasticity Equations
        # FIXED: Removed underscores in tau_pre/tau_post to comply with Brian2 naming rules
        stdp_eqs = '''
            w : 1
            nu : 1 (shared)
            taupre = 20*ms : second
            taupost = 20*ms : second
            dApre/dt = -Apre / taupre : 1 (event-driven)
            dApost/dt = -Apost / taupost : 1 (event-driven)
        '''
        self.synapses = b2.Synapses(self.neurons, self.neurons, model=stdp_eqs,
                                    on_pre='''v_post += w; Apre += 0.01 * nu; w = clip(w + Apost * nu, 0, 1)''',
                                    on_post='''Apost -= 0.01 * nu; w = clip(w + Apre * nu, 0, 1)''',
                                    name='v27_synapses')
        self.synapses.connect(p=0.5)
        self.synapses.w = 'rand() * 0.1'
        self.synapses.nu = 1.0

        self.net = b2.Network(self.neurons, self.synapses)
        self.temporal_window = deque(maxlen=4)

    def process_step(self, coh, sync, concept_id):
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']

        self.net.run(20 * b2.ms)
        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        self.temporal_window.append(list(v_norm) + [sync])

        if len(self.temporal_window) == 4:
            seq_tensor = torch.tensor([list(self.temporal_window)], dtype=torch.float32)
            self.optimizer.zero_grad()
            pred_concept, pred_coh = self.net_model(seq_tensor)

            loss = nn.CrossEntropyLoss()(pred_concept, torch.tensor([concept_id % 31], dtype=torch.long))
            loss.backward()
            self.optimizer.step()

            # Gated Plasticity Feedback
            self.synapses.nu = np.clip(loss.item() / 5.0, 0.05, 1.5)

            # Homeostatic Synaptic Scaling
            if np.mean(self.synapses.w) > 0.4:
                self.synapses.w *= 0.98

            return loss.item()
        return 0

# --- 4. BEHAVIORAL REPORTER ---
def report_behavior(hive, step, avg_loss):
    weights = np.array(hive.synapses.w)
    sparsity = np.sum(weights < 0.1) / len(weights)
    print(f"--- [Step {step:04d}] Behavioral Report ---")
    print(f"  Loss: {avg_loss:.4f} | Avg weight: {np.mean(weights):.3f} | Sparsity: {sparsity:.1%}")
    if sparsity > 0.5: print("  Status: 0 Neural pathways refined.")
    else: print("  Status: 1 Constructing synaptic foundation...")
    print("-" * 35)

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("2 Loading Big Test Dataset (Wikitext)...")
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

    hive = HoloSynV27Hive()
    total_steps = 0
    running_loss = []

    print("\n3 Starting Global Stress-Test (V27)...\n")
    for row in dataset:
        text = row['text'].strip()
        if len(text.split()) < 5: continue

        # Simplified bridge for big test
        words = text.split()
        for i, word in enumerate(words):
            coh = 0.8 if len(word) > 5 else 0.4
            sync = 0.9 if word[-1] in ".,!?" else 0.7

            loss = hive.process_step(coh, sync, concept_id=len(word))
            if loss > 0: running_loss.append(loss)

            total_steps += 1
            if total_steps % 500 == 0:
                report_behavior(hive, total_steps, np.mean(running_loss[-100:]))

            if total_steps >= 5000: break
        if total_steps >= 5000: break

2 Loading Big Test Dataset (Wikitext)...

3 Starting Global Stress-Test (V27)...



WARNING    The object 'v27_synapses' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipykernel_2623/2267741126.py', line 78, in __init__
    self.synapses = b2.Synapses(self.neurons, self.neurons, model=stdp_eqs, [brian2.core.base.unused_brian_object]
WARNING    The object 'v27_neurons' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipykernel_2623/2267741126.py', line 67, in __init__
    self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', refractory=2*b2.ms, name='v27_neurons') [brian2.core.base.unused_brian_object]
WARNING    The obj

--- [Step 0500] Behavioral Report ---
  Loss: 2.3839 | Avg weight: 0.077 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 1000] Behavioral Report ---
  Loss: 2.3059 | Avg weight: 0.092 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 1500] Behavioral Report ---
  Loss: 2.2632 | Avg weight: 0.104 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 2000] Behavioral Report ---
  Loss: 2.4159 | Avg weight: 0.119 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 2500] Behavioral Report ---
  Loss: 2.2677 | Avg weight: 0.129 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 3000] Behavioral Report ---
  Loss: 2.3684 | Avg weight: 0.147 | Sparsity: 90.9%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 

In [17]:
import torch
import numpy as np
import brian2 as b2

def evaluate_recall(trained_hive, sentence):
    print(f"\n🧠 [V27 RECALL TEST] Reading: '{sentence}'")
    print("-" * 65)

    # 1. FREEZE LEARNING
    trained_hive.net_model.eval() # PyTorch eval mode
    trained_hive.synapses.nu = 0.0 # Turn off STDP plasticity

    words = sentence.split()
    if len(words) < 4:
        print("Sentence too short for the temporal window (needs 4+ words).")
        return

    # Reset the biological temporal window for a fresh thought
    trained_hive.temporal_window.clear()

    with torch.no_grad():
        for step, word in enumerate(words):
            # Extract basic features (using the same bridge logic as training)
            coh = 0.8 if len(word) > 5 else 0.4
            sync = 0.9 if word[-1] in ".,!?" else 0.7

            # Pulse the SNN
            for i, name in enumerate(LINGUA_STACK.keys()):
                trained_hive.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
            trained_hive.net.run(20 * b2.ms)

            # Read Brain State
            v_norm = (np.array(trained_hive.neurons.v[:]) - 0.5) * 2.0
            trained_hive.temporal_window.append(list(v_norm) + [sync])

            # Predict once the temporal context buffer is full
            if len(trained_hive.temporal_window) == 4:
                seq_tensor = torch.tensor([list(trained_hive.temporal_window)], dtype=torch.float32)

                # Get Transformer prediction
                pred_concept, pred_coh = trained_hive.net_model(seq_tensor)

                # Get the top 3 most likely concepts
                probabilities = torch.nn.functional.softmax(pred_concept[0], dim=0)
                top3_prob, top3_idx = torch.topk(probabilities, 3)

                # Format the sequence context
                context_words = " ".join(words[step-3:step+1])

                print(f"Context: [... {context_words} ]")
                print(f"  -> Predicted Concept ID: {top3_idx[0].item()} ({top3_prob[0].item():.1%} confidence)")
                print(f"  -> Internal Coherence:   {pred_coh.item():.3f}\n")

# --- Run the Tests ---
test_sentences = [
    "The quantum network learns to understand complex data.",
    "A completely unknown and unexpected sequence occurs here!",
    "Faraday induction allows the swarm to calibrate optic lenses."
]

for text in test_sentences:
    evaluate_recall(hive, text)


🧠 [V27 RECALL TEST] Reading: 'The quantum network learns to understand complex data.'
-----------------------------------------------------------------
Context: [... The quantum network learns ]
  -> Predicted Concept ID: 3 (22.1% confidence)
  -> Internal Coherence:   0.371

Context: [... quantum network learns to ]
  -> Predicted Concept ID: 3 (22.1% confidence)
  -> Internal Coherence:   0.371

Context: [... network learns to understand ]
  -> Predicted Concept ID: 3 (22.1% confidence)
  -> Internal Coherence:   0.371

Context: [... learns to understand complex ]
  -> Predicted Concept ID: 3 (22.1% confidence)
  -> Internal Coherence:   0.371

Context: [... to understand complex data. ]
  -> Predicted Concept ID: 3 (22.1% confidence)
  -> Internal Coherence:   0.371


🧠 [V27 RECALL TEST] Reading: 'A completely unknown and unexpected sequence occurs here!'
-----------------------------------------------------------------
Context: [... A completely unknown and ]
  -> Predicted Concep

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings
from collections import deque
from datasets import load_dataset

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor": {"role": "Perceptron", "weight": 1.0},
    "Seek_Attention": {"role": "Temporal Core", "weight": 1.5},
    "Semantic_Router": {"role": "FFN Bridge",  "weight": 1.2},
    "Lang_Decoder":    {"role": "Causal Gen",  "weight": 1.3}
}

# --- 2. THE V27 TEMPORAL LLM (Transformer-SNN Hybrid) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# --- 1. THE V28 SENSITIVE LLM ---
class HoloSynV28LM(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # V28 Upgrade: 3-Stage Embedding to catch micro-fluctuations
        self.embedding = nn.Sequential(
            nn.Linear(num_nodes + 1, 64),
            nn.GELU(),
            nn.Linear(64, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pos_encoder = PositionalEncoding(hidden_dim)

        # Added Dropout to prevent over-reliance on a single dead path
        decoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True, dropout=0.2)
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=2)

        self.decoder = nn.Linear(hidden_dim, 31)
        self.coherence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x_seq):
        seq_len = x_seq.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x_seq.device)
        x_emb = self.pos_encoder(self.embedding(x_seq))
        context_seq = self.transformer(x_emb, mask=causal_mask, is_causal=True)
        final_context = context_seq[:, -1, :]
        return self.decoder(final_context), self.coherence_head(final_context)

# --- 2. THE V28 BALANCED HIVE ---
class HoloSynV28Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = HoloSynV28LM(self.num_nodes)

        # V28 Upgrade: Added Weight Decay to PyTorch to prevent collapse
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.001, weight_decay=0.01)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 (unless refractory)\n I_in : 1'
        # V28 Upgrade: Added noise to threshold to simulate organic variance
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v > (0.8 + 0.1*randn())', reset='v=0', refractory=2*b2.ms, name='v28_neurons')

        stdp_eqs = '''
            w : 1
            nu : 1 (shared)
            tau_pre = 20*ms : second
            tau_post = 20*ms : second
            dApre/dt = -Apre / tau_pre : 1 (event-driven)
            dApost/dt = -Apost / tau_post : 1 (event-driven)
        '''
        self.synapses = b2.Synapses(self.neurons, self.neurons, model=stdp_eqs,
                                    on_pre='''v_post += w; Apre += 0.01 * nu; w = clip(w + Apost * nu, 0.05, 1)''', # V28: Min weight floor of 0.05
                                    on_post='''Apost -= 0.01 * nu; w = clip(w + Apre * nu, 0.05, 1)''',
                                    name='v28_synapses')
        self.synapses.connect(p=0.5)
        self.synapses.w = 'rand() * 0.2'
        self.synapses.nu = 1.0

        self.net = b2.Network(self.neurons, self.synapses)
        self.temporal_window = deque(maxlen=4)

    def process_step(self, coh, sync, concept_id):
        # Inject Signal
        for i, name in enumerate(LINGUA_STACK.keys()):
            # V28 Upgrade: Added 5% sensory noise to keep inputs dynamic
            noise = np.random.normal(1.0, 0.05)
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight'] * noise

        self.net.run(20 * b2.ms)
        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        self.temporal_window.append(list(v_norm) + [sync])

        if len(self.temporal_window) == 4:
            seq_tensor = torch.tensor([list(self.temporal_window)], dtype=torch.float32)
            self.optimizer.zero_grad()
            pred_concept, pred_coh = self.net_model(seq_tensor)

            loss = nn.CrossEntropyLoss()(pred_concept, torch.tensor([concept_id % 31], dtype=torch.long))
            loss.backward()
            self.optimizer.step()

            self.synapses.nu = np.clip(loss.item() / 5.0, 0.05, 1.5)

            # V28 Upgrade: Soft Subtraction instead of Harsh Multiplication
            if np.mean(self.synapses.w) > 0.4:
                self.synapses.w = np.clip(np.array(self.synapses.w) - 0.01, 0.05, 1.0)

            return loss.item()
        return 0

# --- 4. BEHAVIORAL REPORTER ---
def report_behavior(hive, step, avg_loss):
    weights = np.array(hive.synapses.w)
    sparsity = np.sum(weights < 0.1) / len(weights)
    print(f"--- [Step {step:04d}] Behavioral Report ---")
    print(f"  Loss: {avg_loss:.4f} | Avg weight: {np.mean(weights):.3f} | Sparsity: {sparsity:.1%}")
    if sparsity > 0.5: print("  Status: 0 Neural pathways refined.")
    else: print("  Status: 1 Constructing synaptic foundation...")
    print("-" * 35)

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("2 Loading Big Test Dataset (Wikitext)...")
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

    hive = HoloSynV27Hive()
    total_steps = 0
    running_loss = []

    print("\n3 Starting Global Stress-Test (V27)...\n")
    for row in dataset:
        text = row['text'].strip()
        if len(text.split()) < 5: continue

        # Simplified bridge for big test
        words = text.split()
        for i, word in enumerate(words):
            coh = 0.8 if len(word) > 5 else 0.4
            sync = 0.9 if word[-1] in ".,!?" else 0.7

            loss = hive.process_step(coh, sync, concept_id=len(word))
            if loss > 0: running_loss.append(loss)

            total_steps += 1
            if total_steps % 500 == 0:
                report_behavior(hive, total_steps, np.mean(running_loss[-100:]))

            if total_steps >= 5000: break
        if total_steps >= 5000: break

2 Loading Big Test Dataset (Wikitext)...

3 Starting Global Stress-Test (V27)...

--- [Step 0500] Behavioral Report ---
  Loss: 2.4002 | Avg weight: 0.057 | Sparsity: 100.0%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 1000] Behavioral Report ---
  Loss: 1.8871 | Avg weight: 0.057 | Sparsity: 100.0%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 1500] Behavioral Report ---
  Loss: 1.7432 | Avg weight: 0.057 | Sparsity: 100.0%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 2000] Behavioral Report ---
  Loss: 2.4421 | Avg weight: 0.057 | Sparsity: 100.0%
  Status: 0 Neural pathways refined.
-----------------------------------
--- [Step 2500] Behavioral Report ---
  Loss: 2.2584 | Avg weight: 0.057 | Sparsity: 100.0%
  Status: 0 Neural pathways refined.
-----------------------------------
